## Projeto Onboarding Colaboradores

In [ ]:
# chamamos OS "o diplomata" para comunicar com arquivos do computador, 
# ele vai usar o metodo os.getenv para entrar no sistema operacional e buscar as chaves secretas

import os 

# trazemos o metodo de leitura da biblioteca dotenv o load_dotenv para conseguir acessar nosso 
# arquivo .env no sistema operacional. o OS nosso diplomata precisa da chave certa para abrir 
# o arquivo .env, essa chave se chama load_dotenv
from dotenv import load_dotenv 

# FEYNMAN: O Diplomata (os) usa a chave (load_dotenv) para abrir o cofre (.env) 
# e pegar nossa senha secreta da OpenAI para colocá-la na corrente sanguínea do sistema.
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# chamamos o metodo de leitura desta biblioteca o PyPDFLoader para ler arquivos PDF
# o langchain_community possui recursos uteis desenvolvidos pela comunidade
from langchain_community.document_loaders import PyPDFLoader 

# --- Lendo e Fatiando o Documento ---

# 1. Carregar o documento
# FEYNMAN: PyPDFLoader é o "Tradutor Oficial". Ele pega um arquivo PDF (que o Python não entende) 
# e extrai o texto bruto para que possamos trabalhar com ele.
carregador = PyPDFLoader(r"/seu_caminho_de_pasta/politica_home_office.pdf")
documentos = carregador.load()

In [ ]:
# Usamos o TextSplitter como uma guilhotina inteligente. Por que cortar o texto? Porque o 'estômago' do nosso LLM 
# (a janela de contexto) é pequeno. Ele não consegue engolir um livro de 50 páginas de uma vez e responder perguntas 
# sobre ele. Então, usamos a guilhotina para cortar o livro em pequenos parágrafos (chunks). 
# Assim, o robô consegue ler mastigado, um pedaço de cada vez, sem passar mal.

from langchain_text_splitters import RecursiveCharacterTextSplitter 

# 2. Fatiar o documento
# FEYNMAN: O RecursiveCharacterTextSplitter é a nossa "Guilhotina Cuidadosa". 
# Como a IA tem um "estômago" pequeno, fatiamos o livro em pedaços mastigáveis. 
# A sobreposição (overlap) garante que a guilhotina não corte uma ideia importante no meio.
fatiador = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200
)

In [ ]:
# "O OpenAIEmbeddings é o nosso Cartógrafo. Ele transforma textos em coordenadas de GPS (vetores).
#  Assim, quando o usuário faz uma pergunta, conseguimos calcular a distância matemática para encontrar os 
# parágrafos do PDF que falam sobre o mesmo assunto."

# chamamos chatopenai para conversar com o modelo de linguagem, é o nosso "Intérprete". 
# Ele entende as perguntas do usuário e responde com base no que leu dos parágrafos. 
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 3. Executar o fatiamento
textos_fatiados = fatiador.split_documents(documentos)

# --- Criando o Banco Vetorial ---

# 1. Criar o modelo de embeddings
# FEYNMAN: O OpenAIEmbeddings é o "Cartógrafo". Ele não lê o texto para responder, 
# ele transforma cada palavra/frase em uma coordenada matemática (Vetor) no espaço.
modelo_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# FEYNMAN: FAISS é o nosso "Arquivo Mágico". O Cartógrafo (Embeddings) vai criar as 
# coordenadas de GPS dos textos, mas é o FAISS que funciona como um gaveteiro ultra-rápido. 
# Ele guarda essas coordenadas e consegue calcular a distância entre elas em milissegundos 
# para achar os assuntos parecidos.
from langchain_community.vectorstores import FAISS

# 2. Criar o banco vetorial
# FEYNMAN: Entregamos as fatias de texto e o Cartógrafo para o gaveteiro FAISS guardar tudo.
banco_vetorial = FAISS.from_documents(textos_fatiados, modelo_embeddings)

# 3. Transformando o banco de dados em um Retriever
# FEYNMAN: O "Bibliotecário". Ele é a função que vai entrar no gaveteiro FAISS. 
# O "k=3" é a instrução: "Quando eu perguntar algo, me traga exatamente as 3 páginas mais parecidas".
retriever = banco_vetorial.as_retriever(search_kwargs={"k":3})

In [ ]:
# FEYNMAN: O create_retriever_tool é o "Criador de Uniforme". Ele pega o nosso Bibliotecário
# (retriever) e o veste com um uniforme que o LLM reconhece como uma ferramenta disponível.
# Sem o uniforme, o agente não sabe que pode chamar o Bibliotecário para buscar informações.
from langchain.tools.retriever import create_retriever_tool

ferramenta_rh = create_retriever_tool(
    retriever=retriever,
    name="pesquisa_regras_rh",
    description="Útil para buscar informações sobre regras de Home Office, Férias e Conduta da empresa. Use SEMPRE que o usuário perguntar sobre políticas internas."
    )

In [ ]:
# FEYNMAN: O decorador @tool é o "Crachá de Identificação". Qualquer função Python com esse
# crachá se transforma em uma ferramenta que o agente consegue chamar por conta própria.
# Sem o crachá, o LLM não sabe que aquela função existe ou pode ser usada.
from langchain_core.tools import tool 

# Em seguida, criamos uma ferramenta do zero usando o decorador @tool. Isso diz ao LangChain: "Transforme esta função Python em algo que o LLM entenda".

@tool 
def calculadora_beneficios(salario: float, percentual: float) -> float:
    """
    Calcula o valor do benefício com base no salário e percentual fornecidos.
    
    Args:
        salario (float): O salário base do funcionário.
        percentual (float): O percentual do benefício a ser calculado (ex: 0.1 para 10%).
    
    Returns:
        float: O valor do benefício calculado.
    """
    print(f"DEBUG: O LLM usou a calculadora com salário {salario} e percentual {percentual}!")
    return salario * percentual


In [ ]:
# FEYNMAN: O ChatPromptTemplate é o "Roteiro de Entrevista". Ele define a estrutura da conversa:
# quem fala quando e o que pode variar. O MessagesPlaceholder é o "Espaço Reservado" dentro do
# roteiro — um buraco que será preenchido com o histórico real da conversa em tempo de execução.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Agrupamos as ferramentas em uma lista e criamos o prompt especial para agentes.

lista_de_ferramentas = [ferramenta_rh, calculadora_beneficios]

# 2. Modelo de linguagem
# FEYNMAN: O ChatOpenAI é o "Cérebro" ou "Walkie-Talkie". É quem vai ler o Molde, 
# processar a informação que o Bibliotecário trouxe e formular a resposta em português claro.
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# O Prompt de um agente precisa de um espaço para ele pensar ("agent_scratchpad")
prompt_agente = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de RH da Nexus AI. Use as ferramentas disponíveis para responder o usuário com precisão."),
    MessagesPlaceholder(variable_name="chat_history"), # memoria
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}") # Aqui é onde o gerente anota o uso das ferramentas
])


In [ ]:
# FEYNMAN: O InMemoryChatMessageHistory é o "Caderno de Memórias". Enquanto a sessão está
# ativa, ele anota cada mensagem (humano e IA) para que o agente nunca esqueça o que foi dito.
# É como um bloco de notas que existe só na RAM — apagado quando o programa encerra.
from langchain_core.chat_history import InMemoryChatMessageHistory

# FEYNMAN: O RunnableWithMessageHistory é o "Gerente de Memória". Ele envolve o agente executor
# e fica responsável por buscar o caderno certo (pelo session_id) antes de cada resposta,
# e anotar a nova mensagem logo depois — tudo de forma automática e transparente.
from langchain_core.runnables.history import RunnableWithMessageHistory

memoria_agente = {}
def historico_agente(sessao:str):
    if sessao not in memoria_agente:
        memoria_agente[sessao] = InMemoryChatMessageHistory()
    return memoria_agente[sessao]


In [ ]:
# FEYNMAN: O create_tool_calling_agent é o "Arquiteto do Agente". Ele une o cérebro (LLM),
# as ferramentas e o roteiro em uma lógica coesa. O AgentExecutor é o "Motor" que põe essa
# lógica para rodar, gerenciando o loop: pensar → escolher ferramenta → ver resultado → responder.
from langchain.agents import create_tool_calling_agent, AgentExecutor

# 1. Cria a lógica (Une prompt, LLM e as ferramentas)
agente = create_tool_calling_agent(modelo, lista_de_ferramentas, prompt_agente)

# 2. Cria o executor (O motor que roda a lógica)
agente_executor = AgentExecutor(agent=agente, tools=lista_de_ferramentas, verbose=True)

In [14]:
agente_com_memoria = RunnableWithMessageHistory(
    runnable=agente_executor,
    get_session_history=historico_agente,
    input_messages_key="input",
    history_messages_key="chat_history"
)


In [16]:
# Pergunta 1: Alimentando a máquina
resposta_1 = agente_com_memoria.invoke(
    {"input": "Quero saber quais dias posso trabalhar home office. Além disso, se meu salário base é R$ 8.000 e a empresa me dá 5% desse valor como auxílio para montar o escritório em casa, qual será o valor do meu auxílio?"},
    config={"configurable": {"session_id": "sessao_lucas_rh"}}
)
print("RESPOSTA 1:", resposta_1["output"])

print("\n------------------\n")



> Entering new AgentExecutor chain...

Invoking: `pesquisa_regras_rh` with `{'query': 'dias de trabalho home office'}`


negociado no Adendo do Contrato de modalidade de Home Offi ce.
 
6. JORNADA DE TRABALHO  
A jornada de trabalho em dias de home office deverá seguir as mesmas regras 
estabelecidas para a jornada realizada nos estabelecimentos da Empresa; 
Para os colaboradores que ocupem cargos de confiança e/ou gestão, o trabalho em home 
office não configurará nenhuma forma de controle de jornada, devendo prevalecer o que 
está estabelecido em seus contratos individuais de trabalho;

Limpeza e áreas correlatas. 
 
4. PROCEDIMENTO 
 
Frequência 
 
Deverá ser negociado entre o colaborador e seu gestor imediato , os dias da 
semana que ocorrerão o trabalho home office, podendo ser realizado em qualquer 
dia da semana, de segunda a sexta-feira, observados os horários contratuais, assim 
como a disponibilidade das mesas em aplicativo de reserva e Check in (Desk4Me); 
A frequência de 

In [18]:
# Pergunta 2: O Teste de Amnésia (Não passe o salário nesta pergunta!)
resposta_2 = agente_com_memoria.invoke(
    {"input": "Entendi. E se eu tiver uma redução de 2 mil no salario, qual passa a ser o valor do auxílio?"},
    config={"configurable": {"session_id": "sessao_lucas_rh"}}
)
print("RESPOSTA 2:", resposta_2["output"])



> Entering new AgentExecutor chain...

Invoking: `calculadora_beneficios` with `{'salario': 6000, 'percentual': 0.05}`


DEBUG: O LLM usou a calculadora com salário 6000.0 e percentual 0.05!
300.0Se o seu salário for reduzido para R$ 6.000, o valor do auxílio para montar o escritório em casa passará a ser **R$ 300,00**.

> Finished chain.
RESPOSTA 2: Se o seu salário for reduzido para R$ 6.000, o valor do auxílio para montar o escritório em casa passará a ser **R$ 300,00**.
